# Benchmark LifeMate trên Colab

Chạy theo thứ tự từ trên xuống: **Runtime → Change runtime type → T4 GPU**, rồi chạy từng cell.

1. **Lấy code và kit.** Clone nhánh `colab-smoke` từ GitHub. Kit TRAIN lấy từ Google Drive.
2. **Cài đặt và tải model.** Qwen3-4B-Instruct-2507 và PhoWhisper-medium, chạy offline khi benchmark.
3. **Smoke 40 mẫu.** Chạy planner trên 40 ID TRAIN và chấm với gold.
4. **Tách lỗi ASR khỏi lỗi planner** (tùy chọn). Chạy lại với transcript gốc.
5. **Kết quả** tự lưu lên Drive; có thể tải thêm file zip.

Chưa chạy phần 269 mẫu ở cuối notebook cho tới khi đã xem kết quả smoke.

## 1. Lấy code và kit

**Colab:** tải `train+public_test.zip` lên thư mục gốc Google Drive một lần.
**Kaggle:** bật *Internet* và *GPU T4* ở Settings, rồi *Add Input* dataset chứa kit (`train+public_test`, dạng thư mục hoặc zip). Notebook tự nhận môi trường.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

BRANCH = 'colab-smoke'
REPO_URL = 'https://github.com/Quang1104/lifemate_superklang.git'
ON_KAGGLE = Path('/kaggle/working').is_dir()
BASE = Path('/kaggle/working' if ON_KAGGLE else '/content')
# Colab: file .zip hoặc thư mục kit trên Drive. Kaggle: thêm kit làm Dataset (Add Input), notebook tự tìm.
KIT_ON_DRIVE = Path('/content/drive/MyDrive/train+public_test.zip')
RESULTS_ON_DRIVE = BASE / 'lifemate_results' if ON_KAGGLE else Path('/content/drive/MyDrive/lifemate_results')

REPO = BASE / 'lifemate_superklang'
KIT = Path('/tmp/train+public_test') if ON_KAGGLE else Path('/content/train+public_test')
RUNS = REPO / 'outputs' / 'colab'
MODELS = Path('/tmp/lifemate_models') if ON_KAGGLE else Path('/content/lifemate_models')
# Qwen3.5-4B chạy ~14 token/s trên T4 vì thiếu kernel linear-attention, nên mặc định dùng Qwen3-4B-Instruct-2507.
PLANNER = 'fallback_planner'
# T4 chậm hơn máy chấm. TIME_SCALE > 1 nhân mọi giới hạn thời gian để đo độ chính xác trên TRAIN;
# kết quả khi đó không chứng minh đạt giới hạn 30 giây. Đặt 1 để đo đúng giới hạn thật.
TIME_SCALE = 3

if ON_KAGGLE:
    found = [path for pattern in ('*/train/train.jsonl', '*/*/train/train.jsonl', '*/*/*/train/train.jsonl', '*/*.zip')
             for path in Path('/kaggle/input').glob(pattern)]
    assert found, 'Chưa thấy kit trong /kaggle/input. Bấm Add Input và thêm dataset chứa train+public_test.'
    KIT_ON_DRIVE = found[0] if found[0].suffix == '.zip' else found[0].parent.parent
else:
    from google.colab import drive
    drive.mount('/content/drive')

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], check=True)

if not (KIT / 'train' / 'train.jsonl').is_file():
    assert KIT_ON_DRIVE.exists(), f'Không thấy kit ở {KIT_ON_DRIVE}'
    if KIT_ON_DRIVE.suffix == '.zip':
        unpacked = KIT.parent / 'kit_unzip'
        shutil.rmtree(unpacked, ignore_errors=True)
        # Bỏ metadata AppleDouble (._*, __MACOSX); đó không phải dữ liệu.
        subprocess.run(['unzip', '-q', '-o', str(KIT_ON_DRIVE), '-d', str(unpacked),
                        '-x', '__MACOSX/*', '*/._*', '._*'], check=True)
        roots = [path.parent.parent for pattern in ('train/train.jsonl', '*/train/train.jsonl')
                 for path in unpacked.glob(pattern)]
        assert len(roots) == 1, f'Zip cần đúng một thư mục chứa train/train.jsonl, thấy: {roots}'
        roots[0].rename(KIT)
    else:
        # Chép ra ổ ghi được; bỏ metadata AppleDouble.
        shutil.copytree(KIT_ON_DRIVE, KIT, ignore=shutil.ignore_patterns('._*', '__MACOSX'))
assert (KIT / 'train' / 'train.jsonl').is_file(), f'Kit thiếu train/train.jsonl: {KIT}'
print('Kit:', KIT)

## 2. Cài đặt và tải model

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], cwd=REPO, env=env, check=True)

run('nvidia-smi')
RUNS.mkdir(parents=True, exist_ok=True)
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[model]')
run(sys.executable, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())')

In [ ]:
from huggingface_hub import snapshot_download

pins = json.loads((REPO / 'configs' / 'colab_models.json').read_text())
for role in (PLANNER, 'asr'):
    snapshot_download(**pins[role], local_dir=MODELS / role,
                      allow_patterns=['*.json', '*.jinja', '*.safetensors', '*.bin', 'merges.txt', 'LICENSE', 'README.md'])

offline = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
catalog = KIT / 'train' / 'train_tools_catalog.json'
smoke_manifest = REPO / 'research' / 'smoke_split.json'

def save(target):
    """Chép kết quả lên Drive để không mất khi phiên bị ngắt."""
    destination = RESULTS_ON_DRIVE / PLANNER / target.name
    shutil.copytree(target, destination, dirs_exist_ok=True)
    print('Đã lưu:', destination)

def benchmark(manifest, tag, modes=('native',), transcripts=False):
    """Chạy từng nhánh planner trên manifest, chấm với gold và ghi chi tiết từng mẫu."""
    target = RUNS / PLANNER / f'{tag}_x{TIME_SCALE:g}'
    gold = target / 'gold.jsonl'
    if not gold.is_file():
        run(sys.executable, '-B', '-m', 'scripts.replay_train_gold', '--data-root', KIT,
            '--manifest', manifest, '--output', gold, env=offline)
    source = ['--train-transcripts'] if transcripts else ['--asr-dir', MODELS / 'asr']
    for mode in modes:
        name = mode + ('_oracle' if transcripts else '')
        run(sys.executable, '-B', '-m', 'src.inference', 'official',
            '--split', 'train', '--data-root', KIT, '--train-manifest', manifest,
            '--model-dir', MODELS / PLANNER, *source, '--device', 'cuda',
            '--planner-mode', mode, '--time-scale', TIME_SCALE, '--output', target / f'{name}.jsonl',
            '--train-report', target / f'{name}.report.json', env=offline)
        run(sys.executable, '-B', '-m', 'scripts.score_run', '--manifest', manifest, '--gold', gold,
            '--catalog', catalog, '--predictions', target / f'{name}.jsonl',
            '--report', target / f'{name}.report.json', '--train-jsonl', KIT / 'train' / 'train.jsonl',
            '--output', target / f'{name}.scored.json', env=offline)
        scored = json.loads((target / f'{name}.scored.json').read_text())
        print(name.upper(), round(scored['local_proxy_0_to_100'], 2), scored['lost_proxy_points_by_outcome'],
              'p95', scored['p95_sample_time_ms'], 'statuses', scored.get('statuses'))
        save(target)
    return target

## 3. Smoke 40 mẫu
Nhánh `native` dùng định dạng tool-calling gốc của Qwen. Mỗi nhánh in điểm proxy, điểm mất theo nhóm và lưu `<nhánh>.scored.json` (gold và dự đoán từng mẫu) lên Drive.

In [ ]:
smoke_dir = benchmark(smoke_manifest, 'smoke40', modes=('native',))

## 4. Tách lỗi ASR khỏi lỗi planner (tùy chọn)
Chạy lại với transcript gốc của TRAIN. Chênh lệch điểm so với phần 3 là điểm mất do ASR. Chỉ là chẩn đoán trên TRAIN.

In [ ]:
benchmark(smoke_manifest, 'smoke40', modes=('native',), transcripts=True)

## 6. Fine-tune QLoRA (định dạng native)
Huấn luyện trên 1.236 ID TRAIN ngoài 309 ID validation. Adapter lưu lên Drive mỗi 50 bước; chạy lại cell này sẽ tiếp tục từ bước đã lưu. Sau khi merge, smoke 40 chạy lại với model đã fine-tune.

In [ ]:
SFT_DATA = RUNS / 'sft_native.jsonl'
ADAPTER = RESULTS_ON_DRIVE / 'qlora_native'
MERGED = MODELS / 'planner_sft'
!pip install -q peft bitsandbytes
if not SFT_DATA.is_file():
    !cd {REPO} && python -B -m scripts.build_sft --data-root {KIT} --output {SFT_DATA}
!cd {REPO} && python -B training_code/train_qlora.py --model-dir {MODELS / 'fallback_planner'} --data {SFT_DATA} --output-dir {ADAPTER} --epochs 1 --save-every 25 --max-tokens 12000
!cd {REPO} && python -B training_code/merge_lora.py --model-dir {MODELS / 'fallback_planner'} --adapter-dir {ADAPTER} --output-dir {MERGED}

In [ ]:
PLANNER = 'planner_sft'
smoke_dir = benchmark(smoke_manifest, 'smoke40_sft', modes=('native',))

## 5. Tải kết quả về máy (tùy chọn)
Kết quả đã tự lưu lên Drive sau mỗi nhánh. Cell này chỉ nén và tải thêm một bản về máy.

In [ ]:
archive = shutil.make_archive(str(BASE / f'{PLANNER}_{smoke_dir.name}'), 'zip', smoke_dir)
if ON_KAGGLE:
    print('Tải ở panel Output:', archive)
else:
    from google.colab import files
    files.download(archive)

## 269 mẫu ngoài smoke
Chỉ chạy sau khi đã chốt checkpoint, prompt và thuật toán từ smoke. Không chỉnh cấu hình theo kết quả của phần này.

In [ ]:
locked_manifest = RUNS / 'locked269.json'
run(sys.executable, '-B', '-m', 'scripts.compare_train', 'prepare', '--output', locked_manifest)
locked_dir = benchmark(locked_manifest, 'locked269', modes=('native',))